<a href="https://colab.research.google.com/github/FadouaBOUAFIF/Cours_Traitement_images_TP/blob/main/TP4_Filtrage_Passe_Haut_Contours.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP N°4 : Filtrage Passe-Haut et Détection de Contours sous Python

**Objectifs:**
* Comprendre le rôle des **hautes fréquences** spatiales (variations brusques d'intensité : contours, détails fins et bruit).
* Implémenter les opérateurs différentiels du premier ordre (**Prewitt**, **Sobel**, **Roberts**) par convolution 2D.
* Calculer la **norme du gradient** $G = \sqrt{G_h^2 + G_v^2}$, la **direction du gradient** $\theta = \arctan2(G_h, G_v)$ et appliquer un **seuillage**.
* Comparer la sensibilité au **bruit** des opérateurs du 1er ordre, du **Laplacien / LoG** (2nd ordre) et du détecteur optimal de **Canny** (`skimage.feature.canny`).

---

### 🛠️ 1. Configuration de l'environnement
Exécutez la cellule ci-dessous pour importer les bibliothèques nécessaires et charger l'image de test normalisée entre `[0.0, 1.0]`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, gaussian_laplace
from skimage import data, util, feature, filters

# Chargement de l'image de test en niveaux de gris normalisée [0.0, 1.0]
I = data.camera().astype(np.float64) / 255.0

plt.figure(figsize=(4.5, 4.5))
plt.imshow(I, cmap='gray', vmin=0, vmax=1)
plt.title("Image originale I")
plt.axis('off')
plt.show()

---
## Partie 2 : Étude préliminaire — Gradients directionnels et détection de contours

### Question 2.1 & 2.2 : Gradients vertical ($G_v$), horizontal ($G_h$), norme ($G$) et orientation ($\theta$)
Considérons les deux noyaux suivants :
$$H_v = \begin{pmatrix} -1 & 0 & 1 \\ -1 & 0 & 1 \\ -1 & 0 & 1 \end{pmatrix}, \qquad H_h = \begin{pmatrix} 1 & 1 & 1 \\ 0 & 0 & 0 \\ -1 & -1 & -1 \end{pmatrix}$$

**Travail demandé :**
1. Appliquez ces deux filtres sur `I` pour obtenir $G_v$ et $G_h$. Quels types de contours chaque filtre met-il en évidence ? Que remarquez-vous sur la somme des coefficients de chacun de ces masques ?
2. Calculez la norme du gradient $G = \sqrt{G_h^2 + G_v^2}$ et l'orientation des contours $\theta = \arctan2(G_h, G_v)$ (`np.arctan2(Gh, Gv)`). Affichez les résultats et concluez.

In [ ]:
Hv = np.array([[-1, 0, 1],
               [-1, 0, 1],
               [-1, 0, 1]], dtype=float)

Hh = np.array([[ 1,  1,  1],
               [ 0,  0,  0],
               [-1, -1, -1]], dtype=float)

# À compléter :
# 1. Calculez Gv et Gh avec convolve(I, ..., mode='reflect')

# 2. Calculez la norme G et l'orientation theta (en radians ou degrés)

# 3. Affichez |Gv|, |Gh|, G et theta dans une figure 2x2


**✍️ Vos réponses et déductions (Questions 2.1 & 2.2) :**
> *Double-cliquez ici pour rédiger vos observations sur $G_v$, $G_h$, $G$ et $\theta$...*

---
### Question 2.3 : Filtre de PREWITT
Sous MATLAB, `fspecial('prewitt')` génère le noyau horizontal $h$. Sous Python, nous définissons directement $h = H_h$ et $v = -h^T$.

**Travail demandé :**
1. Exécutez le filtrage de Prewitt avec un seuil `seuil = 0.8` (`Gs = G > seuil`) et commentez le résultat.
2. Faites varier le seuil (par exemple `0.2`, `0.5`, `0.8`, `1.2`) et commentez son influence.
3. Calculez et affichez la somme directe des deux réponses filtrées $G_{somme} = G_h + G_v$. Que déduisez-vous en comparant $G_h + G_v$ avec la norme $G = \sqrt{G_h^2 + G_v^2}$ ?

In [ ]:
h_prewitt = np.array([[ 1,  1,  1],
                      [ 0,  0,  0],
                      [-1, -1, -1]], dtype=float)
v_prewitt = -h_prewitt.T

Gh_p = convolve(I, h_prewitt, mode='reflect')
Gv_p = convolve(I, v_prewitt, mode='reflect')
G_p = np.sqrt(Gh_p**2 + Gv_p**2)

# À compléter :
# 1 & 2. Testez plusieurs seuils [0.2, 0.5, 0.8, 1.2] et affichez Gs = (G_p > seuil)

# 3. Comparez G_somme = Gh_p + Gv_p avec G_p


**✍️ Vos commentaires (Question 2.3 - Prewitt) :**
> *Double-cliquez ici pour commenter l'effet du seuil et le problème de l'addition simple $G_h + G_v$ (annulation des contours diagonaux de signes opposés)...*

---
### Question 2.4 : Filtre de SOBEL
L'opérateur de **Sobel** pondère davantage la ligne ou colonne centrale (facteur `2`) afin d'introduire un léger lissage gaussien orthogonal au gradient :
$$H_{sobel} = \begin{pmatrix} 1 & 2 & 1 \\ 0 & 0 & 0 \\ -1 & -2 & -1 \end{pmatrix}, \qquad V_{sobel} = \begin{pmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{pmatrix}$$

**Travail demandé :**
Appliquez la même démarche que pour Prewitt en ajustant le seuil par le facteur d'échelle des masques : `Gs = G > (seuil * 4 / 3)` (avec `seuil = 0.8`), puis faites varier le seuil et comparez $G_h + G_v$ à $G$.

In [ ]:
h_sobel = np.array([[ 1,  2,  1],
                    [ 0,  0,  0],
                    [-1, -2, -1]], dtype=float)
v_sobel = -h_sobel.T

# À compléter :


**✍️ Vos commentaires (Question 2.4 - Sobel) :**
> *Double-cliquez ici pour rédiger vos commentaires sur Sobel et le facteur 4/3...*

---
### Question 2.5 : Filtre de ROBERTS (Gradients diagonaux $2 \times 2$)
L'opérateur de **Roberts** utilise deux noyaux $2 \times 2$ orientés selon les diagonales ($+45^\circ$ et $-45^\circ$) :
$$a = \begin{pmatrix} 1 & 0 \\ 0 & -1 \end{pmatrix}, \qquad b = \begin{pmatrix} 0 & 1 \\ -1 & 0 \end{pmatrix}$$

**Travail demandé :**
Calculez $G_a$, $G_b$, $G = \sqrt{G_a^2 + G_b^2}$, puis appliquez le seuillage `Gs = G > (seuil * 1 / 3)` (avec `seuil = 0.8`). Répondez aux mêmes questions que précédemment.

In [ ]:
a_roberts = np.array([[ 1,  0],
                      [ 0, -1]], dtype=float)
b_roberts = np.rot90(a_roberts, -1)  # Équivalent de rot90(a, -1) sous MATLAB

# À compléter :


**✍️ Vos commentaires (Question 2.5 - Roberts) :**
> *Double-cliquez ici pour rédiger vos commentaires sur Roberts (finesse des contours vs sensibilité au bruit)...*

---
## Partie 3 : Étude pratique — Comparaison, Bruit et Détecteur de Canny

### Questions 3.1 & 3.2 : Comparaison des détecteurs du 1er ordre et des filtres passe-haut du cours
1. Comparez visuellement les cartes de contours binaires optimales obtenues par **Prewitt**, **Sobel** et **Roberts**.
2. Programmez et appliquez les autres filtres passe-bas/passe-haut vus en cours :
   * **Laplacien** (dérivée seconde isotrope) : $L = \begin{pmatrix} 0 & 1 & 0 \\ 1 & -4 & 1 \\ 0 & 1 & 0 \end{pmatrix}$ ou $L_8 = \begin{pmatrix} 1 & 1 & 1 \\ 1 & -8 & 1 \\ 1 & 1 & 1 \end{pmatrix}$
   * **Rehaussement de contours (*Sharpening*)** : $I_{rehaussée} = I - c \cdot \Delta I$ (masque $\begin{pmatrix} 0 & -1 & 0 \\ -1 & 5 & -1 \\ 0 & -1 & 0 \end{pmatrix}$)
   * **Filtrage Passe-Haut dans le domaine fréquentiel** (complémentaire du passe-bas idéal ou gaussien).

In [ ]:
# À compléter : Comparez Prewitt, Sobel, Roberts, Laplacien et Rehaussement de contours


**✍️ Vos commentaires (Questions 3.1 & 3.2) :**
> *Double-cliquez ici pour comparer les opérateurs du 1er ordre et du 2nd ordre (Laplacien)...*

### Questions 3.3 & 3.4 : Sensibilité au bruit et détecteur optimal de CANNY
1. Ajoutez un bruit Gaussien (`util.random_noise(I, mode='gaussian', var=0.01)`) ou Poivre & Sel (`amount=0.05`) sur l'image `I` et ré-appliquez **Prewitt**, **Sobel** et **Roberts**. Que constatez-vous ?
2. Étudiez et appliquez le détecteur de **Canny** (`feature.canny(image, sigma=1.5, low_threshold=0.1, high_threshold=0.25)`, équivalent de `edge(I, 'canny')` sous MATLAB) sur l'image originale puis sur l'image bruitée. Comparez ses performances avec les opérateurs précédents.

In [ ]:
# Image bruitée
I_bruit = util.random_noise(I, mode='gaussian', var=0.01, rng=42)

# Exemple d'utilisation de Canny sous Python :
contours_canny_propre = feature.canny(I, sigma=1.5)
contours_canny_bruit = feature.canny(I_bruit, sigma=2.0)

# À compléter : Affichez Sobel, Roberts et Canny sur l'image bruitée et comparez


**✍️ Analyse comparative et conclusion générale (Questions 3.3 & 3.4) :**
> *Double-cliquez ici pour expliquer pourquoi les filtres passe-haut amplifient le bruit et pourquoi l'algorithme de Canny (lissage gaussien + gradient + suppression des non-maxima + seuillage par hystérésis) surpasse les détecteurs classiques...*